# DORA RAG — Phase 3 (v2): Embedding & Indexing

Updated for the full 16-document corpus across 3 regulatory levels.

**Changes from v1:**
- Sanity checks now cover Level 1, 2 and 3 documents
- Health check handles multiple docs with overlapping article numbers correctly
- Metadata filter tests cover all doc types: regulation, RTS, ITS, guideline, statement, delegated_regulation
- Index config records regulatory level breakdown
- Batch size guidance updated for larger corpus (~1000+ chunks)

**Prerequisites:** `01_ingestion_v4.ipynb` and `02_chunking.ipynb` must have run successfully.
Expects: `dora_rag/dora_chunks/dora_chunks.json`

In [11]:
# Cell 1 — Install dependencies
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "llama-index",
    "llama-index-embeddings-huggingface",
    "llama-index-vector-stores-chroma",
    "chromadb",
    "sentence-transformers",
    "torch",
    "tqdm",
])
print("Dependencies installed.")

Dependencies installed.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


In [12]:
# Cell 2 — Imports and configuration
import json
import time
import datetime
import torch
from pathlib import Path
from collections import Counter

BASE_DIR    = Path("./dora_rag")
CHUNKS_DIR  = BASE_DIR / "dora_chunks"
CHROMA_DIR  = BASE_DIR / "dora_chroma_db"
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

CHUNKS_FILE = CHUNKS_DIR / "dora_chunks.json"

EMBED_MODEL_NAME  = "BAAI/bge-large-en-v1.5"
EMBED_BATCH_SIZE  = 16    # conservative for larger corpus — increase to 32 if you have 16GB+ RAM
CHROMA_COLLECTION = "dora_regulation"
FORCE_REBUILD     = True  # set True to wipe and rebuild from scratch

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Configuration:")
print(f"  Chunks file      : {CHUNKS_FILE}")
print(f"  Chroma DB path   : {CHROMA_DIR}")
print(f"  Embedding model  : {EMBED_MODEL_NAME}")
print(f"  Embed batch size : {EMBED_BATCH_SIZE}")
print(f"  Device           : {device}")
print(f"  Collection name  : {CHROMA_COLLECTION}")
print(f"  Force rebuild    : {FORCE_REBUILD}")

Configuration:
  Chunks file      : dora_rag/dora_chunks/dora_chunks.json
  Chroma DB path   : dora_rag/dora_chroma_db
  Embedding model  : BAAI/bge-large-en-v1.5
  Embed batch size : 16
  Device           : cpu
  Collection name  : dora_regulation
  Force rebuild    : True


In [13]:
# Cell 3 — Load chunks from Phase 2 and show corpus breakdown
if not CHUNKS_FILE.exists():
    raise FileNotFoundError(
        f"Chunks file not found: {CHUNKS_FILE}\n"
        "Please run 02_chunking.ipynb first."
    )

raw_chunks = json.loads(CHUNKS_FILE.read_text(encoding="utf-8"))
print(f"Loaded {len(raw_chunks)} chunks\n")

# Breakdown by level and doc_type
by_source = Counter(c["source_doc"] for c in raw_chunks)
by_type   = Counter(c["doc_type"]   for c in raw_chunks)

print(f"{'Source document':<45} {'Chunks':>6}")
print("─" * 53)
for src, count in sorted(by_source.items()):
    print(f"  {src[:45]:<45} {count:>6}")
print("─" * 53)
print(f"  {'TOTAL':<45} {len(raw_chunks):>6}")

print(f"\nChunks by doc_type:")
for dtype, count in by_type.most_common():
    print(f"  {dtype:<30}: {count}")

# Sample chunk
s = raw_chunks[0]
print(f"\nSample chunk:")
print(f"  chunk_id    : {s['chunk_id']}")
print(f"  source_doc  : {s['source_doc']}")
print(f"  doc_type    : {s['doc_type']}")
print(f"  article_num : {s['article_num']}")
print(f"  topic_tags  : {s['topic_tags']}")
print(f"  token_est   : {s['token_estimate']}")
print(f"  text preview: {s['text'][:150]}...")

Loaded 390 chunks

Source document                               Chunks
─────────────────────────────────────────────────────
  L1_DORA_Directive_2022_2556                       16
  L1_DORA_Regulation_2022_2554                     116
  L2_DR_CTPP_Designation_Criteria                    7
  L2_DR_DORA_Oversight_Fees                          7
  L2_ITS_ICT_Incident_Reporting                     47
  L2_ITS_Register_of_Information                    47
  L2_RTS_ICT_Incident_Classification                14
  L2_RTS_ICT_Incident_Reporting_Process              8
  L2_RTS_ICT_Risk_Management                        63
  L2_RTS_ICT_Third_Party_Policy                      8
  L2_RTS_Joint_Examination_Team                      7
  L2_RTS_Oversight_Conditions                       12
  L2_RTS_Subcontracting                             10
  L2_RTS_TLPT                                       18
  L3_ESAs_Public_Statement_DORA                      3
  L3_Guidelines_Costs_and_Losses                 

In [14]:
# Cell 4 — Load embedding model
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

print(f"Loading: {EMBED_MODEL_NAME}")
print(f"Device : {device}")
print("First run downloads ~1.3 GB and caches locally.\n")

t0 = time.time()
embed_model = HuggingFaceEmbedding(
    model_name=EMBED_MODEL_NAME,
    query_instruction="Represent this sentence for searching relevant passages: ",
    embed_batch_size=EMBED_BATCH_SIZE,
    device=device,
)
Settings.embed_model = embed_model

print(f"✓ Model loaded in {time.time() - t0:.1f}s")
dim = len(embed_model.get_text_embedding("test"))
print(f"✓ Embedding dimension: {dim}  (expected 1024)")

Loading: BAAI/bge-large-en-v1.5
Device : cpu
First run downloads ~1.3 GB and caches locally.

✓ Model loaded in 2.9s
✓ Embedding dimension: 1024  (expected 1024)


In [15]:
# Cell 5 — Initialise Chroma vector store
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
existing      = [c.name for c in chroma_client.list_collections()]
print(f"Existing collections: {existing or 'none'}")

SKIP_INDEXING = False

if CHROMA_COLLECTION in existing and not FORCE_REBUILD:
    chroma_collection = chroma_client.get_collection(CHROMA_COLLECTION)
    existing_count    = chroma_collection.count()
    print(f"Collection '{CHROMA_COLLECTION}' exists with {existing_count} vectors.")

    if existing_count == len(raw_chunks):
        print(f"✓ Count matches chunk file ({len(raw_chunks)}) — skipping embedding step.")
        SKIP_INDEXING = True
    else:
        print(f"⚠ Count mismatch (index: {existing_count} vs chunks: {len(raw_chunks)}) — rebuilding.")
        chroma_client.delete_collection(CHROMA_COLLECTION)
        chroma_collection = chroma_client.create_collection(
            name=CHROMA_COLLECTION,
            metadata={"hnsw:space": "cosine"}
        )
else:
    if FORCE_REBUILD and CHROMA_COLLECTION in existing:
        print("FORCE_REBUILD=True — deleting existing collection.")
        chroma_client.delete_collection(CHROMA_COLLECTION)
    print(f"Creating collection: '{CHROMA_COLLECTION}'")
    chroma_collection = chroma_client.create_collection(
        name=CHROMA_COLLECTION,
        metadata={"hnsw:space": "cosine"}
    )

vector_store    = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
print(f"\n✓ Chroma ready — collection: '{CHROMA_COLLECTION}'")

Existing collections: ['dora_regulation']
FORCE_REBUILD=True — deleting existing collection.
Creating collection: 'dora_regulation'

✓ Chroma ready — collection: 'dora_regulation'


In [16]:
# Cell 6 — Convert chunks to LlamaIndex TextNodes
#
# Key changes vs v1:
# - Added 'level' to metadata so we can filter by regulatory level (1/2/3)
# - chunk_id is now the stable node ID, preventing duplicate inserts on re-runs

from llama_index.core.schema import TextNode

def chunk_to_node(chunk: dict) -> TextNode:
    return TextNode(
        id_  = chunk["chunk_id"],
        text = chunk["text_with_header"],
        metadata = {
            # Citation fields
            "source_doc":    chunk["source_doc"],
            "doc_type":      chunk["doc_type"],
            "level":         chunk.get("level", 0),          # NEW: regulatory level 1/2/3
            "article_num":   chunk["article_num"] if chunk["article_num"] is not None else -1,
            "article_title": chunk["article_title"],
            "paragraph_idx": chunk["paragraph_idx"],
            # Filter fields (Chroma requires scalar values — lists stored as pipe-separated strings)
            "topic_tags":    "|".join(chunk["topic_tags"]),
            "entity_scope":  "|".join(chunk["entity_scope"]),
            # Diagnostic
            "token_estimate": chunk["token_estimate"],
        },
        excluded_embed_metadata_keys=["paragraph_idx", "token_estimate"],
        excluded_llm_metadata_keys=["token_estimate"],
    )


nodes = [chunk_to_node(c) for c in raw_chunks]
print(f"✓ Converted {len(nodes)} chunks to TextNodes")
print(f"\nSample node:")
print(f"  id      : {nodes[0].id_}")
print(f"  metadata: {nodes[0].metadata}")
print(f"  text    : {nodes[0].text[:120]}...")

✓ Converted 390 chunks to TextNodes

Sample node:
  id      : L1_DORA_Regulation_2__art101__p00
  metadata: {'source_doc': 'L1_DORA_Regulation_2022_2554', 'doc_type': 'regulation', 'level': 1, 'article_num': 101, 'article_title': "of the Treaty on the Functioning of the European Union to horizontal cooperation agreements', as well as with", 'paragraph_idx': 0, 'topic_tags': '', 'entity_scope': '', 'token_estimate': 83}
  text    : Article 101 — of the Treaty on the Functioning of the European Union to horizontal cooperation agreements', as well as w...


In [17]:
# Cell 7 — Embed and index all chunks
#
# With 16 documents this may produce 1000–2000+ chunks.
# Expected time on CPU: 5–15 min. On GPU: under 2 min.
# Progress is shown by LlamaIndex. Chroma persists after each batch —
# safe to interrupt; re-run cell 5 with FORCE_REBUILD=False to resume.

from llama_index.core import VectorStoreIndex

# ── Deduplicate nodes by ID ──────────────────────────────────────────────
seen_ids = set()
unique_nodes = []
for node in nodes:
    if node.id_ not in seen_ids:
        seen_ids.add(node.id_)
        unique_nodes.append(node)

if len(unique_nodes) < len(nodes):
    print(f"⚠ Removed {len(nodes) - len(unique_nodes)} duplicate nodes")
print(f"✓ {len(unique_nodes)} unique nodes ready for indexing")

# ── Build index ──────────────────────────────────────────────────────────
device = "cuda" if __import__('torch').cuda.is_available() else "cpu"
print(f"Embedding {len(unique_nodes)} chunks (batch size {EMBED_BATCH_SIZE}, device: {device})...")
t0 = time.time()

index = VectorStoreIndex(
    unique_nodes,
    storage_context=storage_context,
    embed_model=embed_model,
    show_progress=True,
)

elapsed = time.time() - t0
total   = chroma_collection.count()

print(f"\n✓ Indexing complete")
print(f"  Total vectors stored : {total}")
print(f"  Time elapsed         : {elapsed:.1f}s ({elapsed/len(unique_nodes):.2f}s per chunk)")
print(f"  Index persisted to   : {CHROMA_DIR}")


⚠ Removed 9 duplicate nodes
✓ 381 unique nodes ready for indexing
Embedding 381 chunks (batch size 16, device: cpu)...


Generating embeddings: 100%|██████████| 381/381 [00:57<00:00,  6.66it/s]


✓ Indexing complete
  Total vectors stored : 381
  Time elapsed         : 57.6s (0.15s per chunk)
  Index persisted to   : dora_rag/dora_chroma_db


In [18]:
# Cell 8 — Sanity check queries
#
# Updated for full corpus: tests across Level 1, 2, and 3 documents.
# These are embedding-only checks — no LLM involved yet.

from llama_index.core.retrievers import VectorIndexRetriever

retriever = VectorIndexRetriever(index=index, similarity_top_k=5)

def sanity_check(query: str, expected_sources: list = None):
    print(f"\n{'='*65}")
    print(f"QUERY : {query}")
    if expected_sources:
        print(f"EXPECT: hits from {expected_sources}")
    print("─" * 65)

    results = retriever.retrieve(query)
    retrieved_sources = []

    for i, node in enumerate(results, 1):
        src   = node.metadata.get("source_doc", "?")[:35]
        art   = node.metadata.get("article_num", "?")
        dtype = node.metadata.get("doc_type", "?")
        level = node.metadata.get("level", "?")
        score = node.score
        retrieved_sources.append(node.metadata.get("source_doc", ""))
        print(f"  [{i}] L{level} {dtype:<22} Art {art:<4} score {score:.4f}  {src}")

    if expected_sources:
        hits   = [s for s in expected_sources if any(s in r for r in retrieved_sources)]
        misses = [s for s in expected_sources if not any(s in r for r in retrieved_sources)]
        if hits:   print(f"  ✓ Expected sources found : {hits}")
        if misses: print(f"  ✗ Expected sources MISSING: {misses}")


# Level 1 — main regulation
sanity_check(
    "What are the timelines for reporting major ICT incidents?",
    expected_sources=["L1_DORA_Regulation"]
)

# Level 1 — main regulation
sanity_check(
    "Management body accountability for ICT governance",
    expected_sources=["L1_DORA_Regulation"]
)

# Level 2 — RTS
sanity_check(
    "Criteria for classifying a major ICT incident",
    expected_sources=["L2_RTS_ICT_Incident_Classification"]
)

# Level 2 — RTS
sanity_check(
    "Threat-led penetration testing scope and requirements",
    expected_sources=["L2_RTS_TLPT"]
)

# Level 2 — RTS
sanity_check(
    "Subcontracting ICT services chain risk",
    expected_sources=["L2_RTS_Subcontracting"]
)

# Level 3 — guideline
sanity_check(
    "Estimating aggregate annual costs of ICT incidents",
    expected_sources=["L3_Guidelines_Costs_and_Losses"]
)


QUERY : What are the timelines for reporting major ICT incidents?
EXPECT: hits from ['L1_DORA_Regulation']
─────────────────────────────────────────────────────────────────
  [1] L2 ITS                    Art 5    score 0.7637  L2_ITS_ICT_Incident_Reporting
  [2] L1 regulation             Art 21   score 0.7633  L1_DORA_Regulation_2022_2554
  [3] L2 ITS                    Art 3    score 0.7532  L2_ITS_ICT_Incident_Reporting
  [4] L2 ITS                    Art 9    score 0.7519  L2_ITS_ICT_Incident_Reporting
  [5] L2 ITS                    Art 9    score 0.7514  L2_ITS_ICT_Incident_Reporting
  ✓ Expected sources found : ['L1_DORA_Regulation']

QUERY : Management body accountability for ICT governance
EXPECT: hits from ['L1_DORA_Regulation']
─────────────────────────────────────────────────────────────────
  [1] L1 regulation             Art 5    score 0.7495  L1_DORA_Regulation_2022_2554
  [2] L1 regulation             Art 5    score 0.7251  L1_DORA_Regulation_2022_2554
  [3] L2 RTS    

In [19]:
# Cell 9 — Metadata filter tests
#
# Updated to cover all doc_type values now in the corpus:
# regulation, directive, RTS, ITS, delegated_regulation, guideline, statement
# Also tests the new 'level' filter field.

from llama_index.core.vector_stores import (
    MetadataFilter,
    MetadataFilters,
    FilterOperator,
)

def filter_test(query: str, filters: MetadataFilters, label: str):
    print(f"\n{'─'*65}")
    print(f"FILTER TEST: {label}")
    print(f"Query: {query}")
    fr = VectorIndexRetriever(index=index, similarity_top_k=5, filters=filters)
    results = fr.retrieve(query)
    print(f"Results: {len(results)}")
    for node in results:
        print(f"  L{node.metadata.get('level')} {node.metadata.get('doc_type'):<22} "
              f"Art {node.metadata.get('article_num'):<4} "
              f"{node.metadata.get('source_doc','')[:35]}  score: {node.score:.4f}")
    return results


# Filter by regulatory level
filter_test(
    "ICT risk management obligations",
    MetadataFilters(filters=[MetadataFilter(key="level", value=1, operator=FilterOperator.EQ)]),
    "Level 1 documents only"
)

filter_test(
    "ICT risk management obligations",
    MetadataFilters(filters=[MetadataFilter(key="level", value=2, operator=FilterOperator.EQ)]),
    "Level 2 documents only (RTS/ITS)"
)

filter_test(
    "costs and losses reporting",
    MetadataFilters(filters=[MetadataFilter(key="level", value=3, operator=FilterOperator.EQ)]),
    "Level 3 documents only (guidelines)"
)

# Filter by doc_type
filter_test(
    "incident reporting templates",
    MetadataFilters(filters=[MetadataFilter(key="doc_type", value="ITS", operator=FilterOperator.EQ)]),
    "ITS documents only"
)

filter_test(
    "subcontracting critical ICT services",
    MetadataFilters(filters=[MetadataFilter(key="doc_type", value="RTS", operator=FilterOperator.EQ)]),
    "RTS documents only"
)

# Filter by specific article within a specific doc
filter_test(
    "contractual provisions third-party providers",
    MetadataFilters(filters=[
        MetadataFilter(key="source_doc",  value="L1_DORA_Regulation_2022_2554", operator=FilterOperator.EQ),
        MetadataFilter(key="article_num", value=30,                             operator=FilterOperator.EQ),
    ]),
    "Main regulation Article 30 only"
)


─────────────────────────────────────────────────────────────────
FILTER TEST: Level 1 documents only
Query: ICT risk management obligations
Results: 5
  L1 regulation             Art 5    L1_DORA_Regulation_2022_2554  score: 0.7802
  L1 regulation             Art 16   L1_DORA_Regulation_2022_2554  score: 0.7788
  L1 regulation             Art 9    L1_DORA_Regulation_2022_2554  score: 0.7784
  L1 regulation             Art 28   L1_DORA_Regulation_2022_2554  score: 0.7766
  L1 regulation             Art 28   L1_DORA_Regulation_2022_2554  score: 0.7737

─────────────────────────────────────────────────────────────────
FILTER TEST: Level 2 documents only (RTS/ITS)
Query: ICT risk management obligations
Results: 5
  L2 RTS                    Art 15   L2_RTS_ICT_Risk_Management  score: 0.7958
  L2 RTS                    Art 3    L2_RTS_ICT_Risk_Management  score: 0.7944
  L2 RTS                    Art 31   L2_RTS_ICT_Risk_Management  score: 0.7915
  L2 RTS                    Art 38   L2_RT

[NodeWithScore(node=TextNode(id_='L1_DORA_Regulation_2__art030__p01', embedding=None, metadata={'source_doc': 'L1_DORA_Regulation_2022_2554', 'doc_type': 'regulation', 'level': 1, 'article_num': 30, 'article_title': 'Key contractual provisions', 'paragraph_idx': 1, 'topic_tags': 'third_party_risk|governance|supervision', 'entity_scope': '', 'token_estimate': 761}, excluded_embed_metadata_keys=['paragraph_idx', 'token_estimate'], excluded_llm_metadata_keys=['token_estimate'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text="Article 30 — Key contractual provisions\n\n27.12.2022 EN (i) unrestricted rights of access, inspection and audit by the financial entity, or an appointed third party, and by the\ncompetent authority, and the right to take copies of relevant documentation on-site if they are critical to the\noperations of the ICT third-party service provider, the effective exercise of which is not impeded or limited by\nother contractual arrangement

In [21]:
# Cell 10 — Index health check
#
# Key fix vs v1: article gap detection is now per-document rather than
# global, because multiple documents each restart from Article 1.
# A 'gap' in global article numbers is expected and normal.

all_metadata = chroma_collection.get(include=["metadatas"])["metadatas"]

print("=" * 65)
print("INDEX HEALTH CHECK")
print("=" * 65)
print(f"Vectors in index : {chroma_collection.count()}")
print(f"Expected chunks  : {len(raw_chunks)}")
status = "✓ MATCH" if chroma_collection.count() == len(raw_chunks) else "✗ MISMATCH"
print(f"Status           : {status}")

# By level
levels = Counter(m.get("level", 0) for m in all_metadata)
print(f"\nVectors by level:")
for lv in sorted(levels):
    print(f"  Level {lv}: {levels[lv]} vectors")

# By doc_type
doc_types = Counter(m.get("doc_type", "unknown") for m in all_metadata)
print(f"\nVectors by doc_type:")
for dtype, count in doc_types.most_common():
    print(f"  {dtype:<30}: {count}")

# Per-document article coverage (fixes the false-gap-detection bug)
print(f"\nPer-document article coverage:")
print(f"  {'Source doc':<45} {'Arts':>4} {'Range'}")
print("  " + "─" * 62)

from collections import defaultdict
doc_articles = defaultdict(set)
for m in all_metadata:
    art = m.get("article_num", -1)
    src = m.get("source_doc", "unknown")
    if art != -1:
        doc_articles[src].add(art)

for src, arts in sorted(doc_articles.items()):
    if arts:
        art_min = min(arts)
        art_max = max(arts)
        expected = set(range(art_min, art_max + 1))
        gaps     = expected - arts
        gap_str  = f" ⚠ gaps: {sorted(gaps)}" if gaps else " ✓"
        print(f"  {src[:45]:<45} {len(arts):>4}  Art {art_min}–{art_max}{gap_str}")

# Embedding dimension check
sample_vec = chroma_collection.get(limit=1, include=["embeddings"])["embeddings"]
if len(sample_vec) > 0:
    print(f"\nEmbedding dimension: {len(sample_vec[0])} (expected 1024)")


INDEX HEALTH CHECK
Vectors in index : 381
Expected chunks  : 390
Status           : ✗ MISMATCH

Vectors by level:
  Level 1: 130 vectors
  Level 2: 241 vectors
  Level 3: 10 vectors

Vectors by doc_type:
  RTS                           : 133
  regulation                    : 114
  ITS                           : 94
  directive                     : 16
  delegated_regulation          : 14
  guideline                     : 7
  statement                     : 3

Per-document article coverage:
  Source doc                                    Arts Range
  ──────────────────────────────────────────────────────────────
  L1_DORA_Directive_2022_2556                     12  Art 1–18 ⚠ gaps: [12, 13, 14, 15, 16, 17]
  L1_DORA_Regulation_2022_2554                    65  Art 1–101 ⚠ gaps: [65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100]
  L2_DR_CTPP_Designation_Criteria                  7  Art 1–7 ✓
  L

In [22]:
# Cell 11 — Save index config for Phase 4

# Rebuild doc_types and levels for config (in case cell 10 wasn't run)
all_meta_for_config = chroma_collection.get(include=["metadatas"])["metadatas"]
doc_types_config    = dict(Counter(m.get("doc_type", "unknown") for m in all_meta_for_config))
levels_config       = dict(Counter(m.get("level", 0)           for m in all_meta_for_config))

index_config = {
    "created_at":        datetime.datetime.now().isoformat(),
    "embed_model":       EMBED_MODEL_NAME,
    "embed_dimension":   1024,
    "embed_batch_size":  EMBED_BATCH_SIZE,
    "similarity_metric": "cosine",
    "device":            device,
    "chroma_dir":        str(CHROMA_DIR),
    "chroma_collection": CHROMA_COLLECTION,
    "total_vectors":     chroma_collection.count(),
    "source_chunks_file": str(CHUNKS_FILE),
    "corpus_levels":     levels_config,       # NEW: breakdown by regulatory level
    "doc_types_indexed": doc_types_config,
    "notes": (
        "Full 3-level DORA corpus (16 documents). "
        "BGE models require query_instruction prefix at retrieval time. "
        "topic_tags and entity_scope stored as pipe-separated strings. "
        "'level' metadata field enables filtering by regulatory level (1=regulation, 2=RTS/ITS, 3=guidelines)."
    )
}

config_path = BASE_DIR / "index_config.json"
config_path.write_text(json.dumps(index_config, indent=2), encoding="utf-8")
print(f"✓ Index config saved: {config_path}")
print()
print(json.dumps(index_config, indent=2))

✓ Index config saved: dora_rag/index_config.json

{
  "created_at": "2026-04-07T13:24:57.625855",
  "embed_model": "BAAI/bge-large-en-v1.5",
  "embed_dimension": 1024,
  "embed_batch_size": 16,
  "similarity_metric": "cosine",
  "device": "cpu",
  "chroma_dir": "dora_rag/dora_chroma_db",
  "chroma_collection": "dora_regulation",
  "total_vectors": 381,
  "source_chunks_file": "dora_rag/dora_chunks/dora_chunks.json",
  "corpus_levels": {
    "1": 130,
    "2": 241,
    "3": 10
  },
  "doc_types_indexed": {
    "regulation": 114,
    "directive": 16,
    "RTS": 133,
    "ITS": 94,
    "delegated_regulation": 14,
    "guideline": 7,
    "statement": 3
  },
  "notes": "Full 3-level DORA corpus (16 documents). BGE models require query_instruction prefix at retrieval time. topic_tags and entity_scope stored as pipe-separated strings. 'level' metadata field enables filtering by regulatory level (1=regulation, 2=RTS/ITS, 3=guidelines)."
}


In [23]:
# Cell 12 — Phase 3 summary
print("\n" + "=" * 65)
print("PHASE 3 COMPLETE — EMBEDDING & INDEXING")
print("=" * 65)
print(f"""
Embedding model  : {EMBED_MODEL_NAME}
Vector dimension : 1024
Similarity       : cosine
Vector store     : ChromaDB (local persistent)
Collection       : {CHROMA_COLLECTION}
Total vectors    : {chroma_collection.count()}
Index location   : {CHROMA_DIR}

Corpus coverage:
  Level 1 (regulation/directive) : {levels_config.get(1, 0)} vectors
  Level 2 (RTS/ITS)              : {levels_config.get(2, 0)} vectors
  Level 3 (guidelines)           : {levels_config.get(3, 0)} vectors

Outputs:
  {CHROMA_DIR}/   ← persistent vector index
  {config_path}   ← index config for Phase 4

Next step → open 04_retrieval.ipynb
""")


PHASE 3 COMPLETE — EMBEDDING & INDEXING

Embedding model  : BAAI/bge-large-en-v1.5
Vector dimension : 1024
Similarity       : cosine
Vector store     : ChromaDB (local persistent)
Collection       : dora_regulation
Total vectors    : 381
Index location   : dora_rag/dora_chroma_db

Corpus coverage:
  Level 1 (regulation/directive) : 130 vectors
  Level 2 (RTS/ITS)              : 241 vectors
  Level 3 (guidelines)           : 10 vectors

Outputs:
  dora_rag/dora_chroma_db/   ← persistent vector index
  dora_rag/index_config.json   ← index config for Phase 4

Next step → open 04_retrieval.ipynb

